In [1]:
import duckdb
con = duckdb.connect("../data/capstone_data.duckdb")


### Sanity Check of data and columns

In [8]:
con.execute("DESCRIBE customers").df()

,column_name,column_type,null,key,default,extra
0,CustomerID,VARCHAR,YES,None,None,None
1,Individual,VARCHAR,YES,None,None,None
2,Age,DOUBLE,YES,None,None,None
3,Gender,VARCHAR,YES,None,None,None
4,NAICSCode,DOUBLE,YES,None,None,None
5,OriginalCustomerDate,DATE,YES,None,None,None
6,RelationshipYears,DOUBLE,YES,None,None,None
7,RelationshipMonths,DOUBLE,YES,None,None,None
8,BangorWealth,BOOLEAN,YES,None,None,None
9,Payroll,BOOLEAN,YES,None,None,None


In [5]:
# Get column names dynamically
cols = con.execute("DESCRIBE customers").df()['column_name'].tolist()

# Querying missing values and looping through all columns
null_checks = ', '.join([f"COUNT(*) - COUNT({c}) AS {c}" for c in cols])
missing = con.execute(f"SELECT {null_checks} FROM customers").fetchone()

for col, nulls in zip(cols, missing):
    if nulls > 0:
        print(f"  {col}: {nulls:,} missing")

  NAICSCode: 234,887 missing
  OriginalCustomerDate: 24 missing
  RelationshipYears: 24 missing
  RelationshipMonths: 24 missing


### Customers Overview

In [6]:
query = con.execute("""
                    SELECT 
                    COUNT(*) AS total_customers,
                    COUNT(distinct CustomerID) AS unique_customers,
                    SUM(CASE WHEN Individual = 'Y' THEN 1 ELSE 0 END) AS individuals,
                    SUM(CASE WHEN Individual = 'N' THEN 1 ELSE 0 END) AS non_individuals
                    FROM customers
                    """).fetchone()

print(f"Total customer rows: {query[0]:,}")
print(f"Total unique customers: {query[1]:,}")
print(f"Total individual customers: {query[2]:,}")
print(f"Total non-individual customers: {query[3]:,}")

Total customer rows: 276,838
Total unique customers: 276,837
Total individual customers: 236,428
Total non-individual customers: 40,410


### Age Distribution

In [7]:
query = con.execute("""
    SELECT 
        COUNT(*) AS total,
        COUNT(Age) AS with_age,
        ROUND(MIN(Age), 1) AS min_age,
        ROUND(MAX(Age), 1) AS max_age,
        ROUND(AVG(Age), 1) AS avg_age,
        ROUND(MEDIAN(Age), 1) AS median_age
    FROM customers
""").fetchone()

print(f"Customers with age: {query[1]:,} / {query[0]:,}")
print(f"Min age:    {query[2]}")
print(f"Max age:    {query[3]}")
print(f"Mean age :   {query[4]}")
print(f"Median age: {query[5]}")

Customers with age: 276,838 / 276,838
Min age:    0.0
Max age:    126.0
Mean age :   41.9
Median age: 43.0


In [3]:
query = con.execute("""
    SELECT *
    FROM customers
    WHERE Age > 100
    ORDER BY Age DESC
""").df()

print(query.T)

                                            0                    1    \
CustomerID                            BSB505021            BSB104853   
Individual                                    Y                    Y   
Age                                       126.0                126.0   
Gender                                                                 
NAICSCode                                   NaN                  NaN   
OriginalCustomerDate        1982-01-21 00:00:00  1970-06-29 00:00:00   
RelationshipYears                          44.0                 56.0   
RelationshipMonths                        528.0                667.0   
BangorWealth                              False                False   
Payroll                                   False                False   
Merchant                                  False                False   
TPS                                       False                False   
PlingCustomer                             False                F

### Analyzing Relationships

In [8]:
query = con.execute("""
    SELECT 
        ROUND(MIN(RelationshipYears), 1) AS min_years,
        ROUND(MAX(RelationshipYears), 1) AS max_years,
        ROUND(AVG(RelationshipYears), 1) AS avg_years,
        ROUND(MEDIAN(RelationshipYears), 1) AS median_years
    FROM customers
""").fetchone()

print(f"Min rel age:    {query[0]} years")
print(f"Max rel age:    {query[1]} years")
print(f"Mean rel age:   {query[2]} years")
print(f"Median rel age: {query[3]} years")

Min rel age:    0.0 years
Max rel age:    85.0 years
Mean rel age:   13.3 years
Median rel age: 10.0 years


### Various products and holdings

In [9]:
query = con.execute("""
    SELECT 
        SUM(CASE WHEN ActiveDebitCard = 'Y' THEN 1 ELSE 0 END) AS active_debit,
        SUM(CASE WHEN ActiveATMCard = 'Y' THEN 1 ELSE 0 END) AS active_atm,
        SUM(CASE WHEN DepositAccount = 'Y' THEN 1 ELSE 0 END) AS deposit_acct,
        SUM(CASE WHEN LoanAccount = 'Y' THEN 1 ELSE 0 END) AS loan_acct,
        SUM(CASE WHEN CreditCardAccount = 'Y' THEN 1 ELSE 0 END) AS credit_card,
        SUM(CASE WHEN TimeDepositAccount = 'Y' THEN 1 ELSE 0 END) AS time_deposit,
        COUNT(*) AS total
    FROM customers
""").fetchone()

total = query[6]
print(f"Active Debit Card:    {query[0]:,} ({query[0]/total*100:.1f}%)")
print(f"Active ATM Card:      {query[1]:,} ({query[1]/total*100:.1f}%)")
print(f"Deposit Account:      {query[2]:,} ({query[2]/total*100:.1f}%)")
print(f"Loan Account:         {query[3]:,} ({query[3]/total*100:.1f}%)")
print(f"Credit Card Account:  {query[4]:,} ({query[4]/total*100:.1f}%)")
print(f"Time Deposit Account: {query[5]:,} ({query[5]/total*100:.1f}%)")

Active Debit Card:    180,912 (65.3%)
Active ATM Card:      8,230 (3.0%)
Deposit Account:      260,163 (94.0%)
Loan Account:         57,175 (20.7%)
Credit Card Account:  6,391 (2.3%)
Time Deposit Account: 14,336 (5.2%)


### Checking boolean value columns - 

In [10]:
query = con.execute("""
    SELECT 
        SUM(CASE WHEN TPS = true THEN 1 ELSE 0 END) AS tps,
        SUM(CASE WHEN Payroll = true THEN 1 ELSE 0 END) AS payroll,
        SUM(CASE WHEN Merchant = true THEN 1 ELSE 0 END) AS merchant,
        SUM(CASE WHEN BangorWealth = true THEN 1 ELSE 0 END) AS bangor_wealth,
        SUM(CASE WHEN PlingCustomer = true THEN 1 ELSE 0 END) AS pling,
        SUM(CASE WHEN ABLECustomer = true THEN 1 ELSE 0 END) AS able,
        COUNT(*) AS total
    FROM customers
""").fetchone()

total = query[6]
print(f"TPS (Treasury and Payment Services): {query[0]:,} ({query[0]/total*100:.1f}%)")
print(f"Payroll:                             {query[1]:,} ({query[1]/total*100:.1f}%)")
print(f"Merchant:                            {query[2]:,} ({query[2]/total*100:.1f}%)")
print(f"BangorWealth:                        {query[3]:,} ({query[3]/total*100:.1f}%)")
print(f"PlingCustomer:                       {query[4]:,} ({query[4]/total*100:.1f}%)")
print(f"ABLECustomer:                        {query[5]:,} ({query[5]/total*100:.1f}%)")

TPS (Treasury and Payment Services): 2,872 (1.0%)
Payroll:                             5,066 (1.8%)
Merchant:                            1,257 (0.5%)
BangorWealth:                        2,239 (0.8%)
PlingCustomer:                       625 (0.2%)
ABLECustomer:                        1,930 (0.7%)


### Gender distribution

In [11]:
query = con.execute("""
    SELECT Gender, COUNT(*) as count
    FROM customers
    GROUP BY Gender
    ORDER BY count DESC
""").df()

print(query)

  Gender   count
0      M  104030
1      F  101410
2          70445
3      O     936
4      U      17


In [12]:
con.execute("""
    SELECT Individual, Gender, COUNT(*) as count
    FROM customers
    GROUP BY Individual, Gender
    ORDER BY Individual, count DESC
""").df()

,Individual,Gender,count
0,N,,39542
1,N,M,839
2,N,F,20
3,N,O,9
4,Y,M,103191
5,Y,F,101390
6,Y,,30903
7,Y,O,927
8,Y,U,17


### Types of accounts and loans

In [13]:
query = con.execute("""
    SELECT 
        ROUND(AVG(NumberActiveDDAs), 2) AS avg_ddas,
        ROUND(AVG(NumberActiveTimeDeposits), 2) AS avg_time_deposits,
        ROUND(AVG(NumberActiveLoans), 2) AS avg_loans,
        ROUND(AVG(NumberCreditCardAccts), 2) AS avg_credit_cards,
        MAX(NumberActiveDDAs) AS max_ddas,
        MAX(NumberActiveTimeDeposits) AS max_time_deposits,
        MAX(NumberActiveLoans) AS max_loans,
        MAX(NumberCreditCardAccts) AS max_credit_cards
    FROM customers
""").fetchone()

print(f"{'Account Type':<25} {'Avg':<10} {'Max':<10}")
print(f"{'Active DDAs':<25} {query[0]:<10} {int(query[4]):<10}")
print(f"{'Active Time Deposits':<25} {query[1]:<10} {int(query[5]):<10}")
print(f"{'Active Loans':<25} {query[2]:<10} {int(query[6]):<10}")
print(f"{'Credit Card Accounts':<25} {query[3]:<10} {int(query[7]):<10}")

Account Type              Avg        Max       
Active DDAs               1.87       227       
Active Time Deposits      0.08       27        
Active Loans              0.27       15        
Credit Card Accounts      0.02       2         


In [14]:
query = con.execute("""
                    SELECT CustomerID, NumberActiveDDAs
                    FROM customers
                    WHERE NumberActiveDDAs = 227
                    """).df()

print(query)

  CustomerID  NumberActiveDDAs
0  BSB343401             227.0


In [15]:
query = con.execute("""
                    SELECT *
                    FROM customers
                    WHERE CustomerID = 'BSB343401'            
                    """).df()

print(query.T)

                                              0
CustomerID                            BSB343401
Individual                                    N
Age                                         0.0
Gender                                         
NAICSCode                              813410.0
OriginalCustomerDate        1982-01-20 00:00:00
RelationshipYears                          44.0
RelationshipMonths                        528.0
BangorWealth                              False
Payroll                                   False
Merchant                                  False
TPS                                       False
PlingCustomer                             False
ABLECustomer                              False
TimeDepositAccount                            N
DepositAccount                                Y
LoanAccount                                   N
CreditCardAccount                             N
ActiveATMCard                                 N
ActiveDebitCard                         

### Primary Banking Customer Flag

In [16]:
query = con.execute("""
    SELECT PrimaryBankingCustomerFlag, COUNT(*) as count
    FROM customers
    GROUP BY PrimaryBankingCustomerFlag
    ORDER BY count DESC
""").df()

print(query)

  PrimaryBankingCustomerFlag   count
0                          Y  176425
1                          N  100413


### Oldest Customers

In [17]:
query = con.execute("""
    SELECT CustomerID, OriginalCustomerDate, RelationshipYears, Age, Individual
    FROM customers
    ORDER BY Age DESC
    LIMIT 25
""").df()

print(query)

   CustomerID OriginalCustomerDate  RelationshipYears    Age Individual
0   BSB505021           1982-01-21               44.0  126.0          Y
1   BSB104853           1970-06-29               56.0  126.0          Y
2   BSB197835           2019-04-10                7.0  126.0          N
3   BSB152620           2010-05-12               16.0  126.0          N
4   BSB179775           2004-01-22               22.0  125.0          N
5   BSB036562           1998-12-01               28.0  125.0          N
6   BSB455643           1992-08-21               34.0  120.0          Y
7   BSB170153           1996-10-03               30.0  118.0          Y
8   BSB169978           1989-11-08               37.0  118.0          Y
9   BSB368581           1977-03-21               49.0  115.0          Y
10  BSB160295           1986-05-21               40.0  114.0          Y
11  BSB323919           1985-06-17               41.0  114.0          Y
12  BSB383415           1969-01-01               57.0  114.0    

In [ ]:
query = con.execute("""
    SELECT *
    FROM customers
    WHERE Age > 100
    ORDER BY Age DESC
""").df()

print(query)

In [18]:
query = con.execute("""
    SELECT *
    FROM customers
    WHERE CustomerID = 'BSB505021'
""").df()

print(query.T)

                                              0
CustomerID                            BSB505021
Individual                                    Y
Age                                       126.0
Gender                                         
NAICSCode                                   NaN
OriginalCustomerDate        1982-01-21 00:00:00
RelationshipYears                          44.0
RelationshipMonths                        528.0
BangorWealth                              False
Payroll                                   False
Merchant                                  False
TPS                                       False
PlingCustomer                             False
ABLECustomer                              False
TimeDepositAccount                            N
DepositAccount                                Y
LoanAccount                                   N
CreditCardAccount                             N
ActiveATMCard                                 N
ActiveDebitCard                         